In [ ]:
from matplotlib.pyplot import show, subplots, tight_layout
from numpy import abs, array, eye, linspace, max, mean, min, sin
from numpy.linalg import eigvals, norm
from numpy.typing import NDArray
from scipy.linalg import expm
from sympy import Function, symbols

a) Zeigen, dass die Trafos kanonisch sind (Poisson-Klammern müssen identisch sein).

In [ ]:
def trafo1(qp, f):
    [q, p] = qp
    return [q + f(p), p]

def trafo2(qp, g):
    [q, p] = qp
    return [q , p + g(q)]

def poisson(u, v, q, p):
    du_dq = u.diff(q)
    du_dp = u.diff(p)
    dv_dq = v.diff(q)
    dv_dp = v.diff(p)
    return du_dq * dv_dp - du_dp * dv_dq

q, p = symbols('q p')
f, g = symbols('f g', cls=Function)

poisson_original = poisson(q, p, q, p)
print(f"Original Poisson bracket: {poisson_original}")

for trafo, func in zip([trafo1, trafo2], [f, g]):
    [xi, eta] = trafo([q, p], func)
    poisson_trafo = poisson(xi, eta, q, p)
    print(f"Transformed Poisson bracket: {poisson_trafo}")


b) Wachstumsrate des Eulerverfahrens am Beispiel des harmonischen Oszillators.

In [ ]:
def HarmOsc_Exact(state: NDArray, t, m: float = 1.0, ω: float = 1.0) -> NDArray:
    A = expm(t * array([[0, 1/m], [-m*ω**2, 0]]))
    return A @ state

def HarmOsc_Euler(state: NDArray, t, m: float = 1.0, ω: float = 1.0) -> NDArray:
    A = eye(2) + t * array([[0, 1/m], [-m*ω**2, 0]])
    return A @ state

def GrowthRate(delta_t, steps):
    qp_exact = qp_euler = [1.0, 0.0]  # Arbitrary initial conditions
    norm_euler = []; norm_exact = []

    for _ in range(steps):
        qp_exact = HarmOsc_Exact(qp_exact, delta_t)
        qp_euler = HarmOsc_Euler(qp_euler, delta_t)

        norm_exact.append(norm(qp_exact))
        norm_euler.append(norm(qp_euler))

    growth_rate_euler = max(norm_euler) / min(norm_euler)
    growth_rate_exact = max(norm_exact) / min(norm_exact)

    return growth_rate_euler, growth_rate_exact

delta_t = 0.01
growth_rate_euler, growth_rate_exact = GrowthRate(delta_t, 1000)
print(f"Wachstumsrate (exakt): {growth_rate_exact}")
print(f"Wachstumsrate (Euler): {growth_rate_euler}")

c) Verkettung sequentieller Updates und Bestimmung der Eigenwerte der entsprechenden Matrix.

In [ ]:
eps = 0.01
m = 1.0
ω = 1.0

C1 = array([[1, eps/m], [0, 1]])
C2 = array([[1, 0], [-eps*m*ω**2, 1]])
C = C2 @ C1

eig = eigvals(C)
print(f"Eigenwerte von C: {eig} mit den Beträgen {abs(eig)}")

d) Verlet-Algorithmus am mathematischen Pendel.

In [ ]:
def C1(state, t, m=1.0):
    return array([[1, t/m], [0, 1]]) @ state

def C2(state, t):
    q, p = state
    V_prime = sin(q)
    return array([q, p - t*V_prime])

def Verlet(state, t):
    state = C1(state, t/2)
    state = C2(state, t)
    state = C1(state, t/2)
    return state

e) Test der Verlet-Methode

In [ ]:
def simulate_verlet(delta_t, steps, ax):
    state = array([1.0, 0.0])  # Initial conditions
    q_result = []
    p_result = []

    for _ in range(steps):
        state = Verlet(state, delta_t)
        q_result.append(state[0])
        p_result.append(state[1])

    return q_result, p_result

def FindPeriod(q, delta_t):
    peaks = []
    for i in range(1, len(q) - 1):
        if q[i-1] < q[i] and q[i] > q[i+1]:
            peaks.append(i)

    periods = []
    for i in range(1, len(peaks)):
        periods.append((peaks[i] - peaks[i-1]) * delta_t)

    if len(periods) > 0:
        avg_period = mean(periods)
    else:
        avg_period = 0

    return avg_period

_, axs = subplots(2, 2)
for ax, dt in zip(axs.flat, [0.1, 0.5, 1.0, 3.0]):
    q, p = simulate_verlet(dt, int(1000/dt), ax)
    period = FindPeriod(q, dt)
    ax.plot(q, p, 'bo', markersize=0.5)
    ax.set_title(f"Verlet mit Δt = {dt} und ω ≈ {period:.2f}")
    ax.set_xlabel("q")
    ax.set_ylabel("p")

periods = []
delta_ts = linspace(0.01, 3.0, 100)
for dt in delta_ts:
    q, p = simulate_verlet(dt, int(1000/dt), ax)
    period = FindPeriod(q, dt)
    periods.append(period)

_, axs2 = subplots()
axs2.plot(delta_ts, periods, 'ro-', markersize=0.5)
axs2.set_title(f"Periodendauer ω über Zeitschritt Δt")
axs2.set_xlabel("Δt")
axs2.set_ylabel("ω")
axs2.set_ylim(0, 10)

tight_layout()
show()